# Case Study v2.1 Inpainting Workflow

This notebook uses the LC-GPT model to inpaint masked NLCD regions with the multiresolution scheme introduced in `case-study-v2-1.py`:
- perform coarse-to-fine inpainting while constraining masked coverage per level
- restrict fine-level logits based on coarse majority votes
- iteratively sample context windows that satisfy the filled-pixel threshold


In [6]:

from dataclasses import dataclass, field
from pathlib import Path
import logging
import shutil
from datetime import datetime
import sys
import numpy as np
import torch
import rasterio
from rasterio.transform import Affine, array_bounds
import rasterio.windows
import matplotlib.pyplot as plt
import geopandas as gpd
from rasterio.features import rasterize as rio_rasterize
from rasterio.warp import Resampling, calculate_default_transform, reproject
from omegaconf import OmegaConf
import pickle
import subprocess
from scipy.stats import mode as sp_mode

sys.path.append("./")

from RandAR.utils import instantiate_from_config
from RandAR.utils.inpainting import generate_inpainting
from RandAR.dataset.nlcd_dataset import detokenize
from RandAR.model.nlcd_tokenizer import NLCDTokenizer


@dataclass
class Config:
    config_yaml: str = "../configs/randar_nlcd_128_large.yaml"
    ckpt_dir: str = "../results/randar_nlcd_128_large/checkpoints/iter_180000"
    data_npz: str = "../data/data_128_final.npz"
    geojson_dir: str = "../data/inpaint_regions"
    nlcd_img_path: str = "../data/nlcd_2021_land_cover_l48_20230630.img"
    case_study_root: str = "../results/case_study"
    bases: list[str] = field(default_factory=lambda: [
        "ft_belvoir",
        "ft_custer_training_center",
        "ft_hood",
        "eglin_afb",
        "joint_base_lewis-mcchord",
        "vandenberg_afb",
    ])
    
    device: str = "cuda"
    window_tokens: int = 64
    stride_tokens: int = 64
    temperature: float = 1.0
    top_k: int = 0
    top_p: float = 1.0
    cfg_scales: tuple[float, float] = (1.0, 1.0)
    seed: int = 42
    forbidden_nlcd: tuple[int, ...] = (11, 12, 90, 95)
    max_mask_ratio_coarse: float = 0.35
    max_mask_ratio_fine: float = 0.70
    min_filled_ratio: float = 0.65
    finest_resolution: int = 120
    test_mode: bool = False
    samples_per_base: int = 10


cfg = Config()
if cfg.test_mode:
    cfg.samples_per_base = 3


In [ ]:

logging.basicConfig(
    level=print,
    format='[%(asctime)s] %(message)s',
    datefmt='%H:%M:%S'
)
print("Initialized configuration")


### Create output folders

In [ ]:

case_root = Path(cfg.case_study_root)
case_root.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir = case_root / f"case-study-v2-1_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=True)

inpaint_dir = run_dir / "inpainted"
consensus_dir = run_dir / "consensus"
pickle_dir = run_dir / "pickles"
figures_dir = run_dir / "figures"

for folder in (inpaint_dir, consensus_dir, pickle_dir, figures_dir):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Using device: {cfg.device} with checkpoint {cfg.ckpt_dir}. Writing outputs to {run_dir}")


### Check for bases

In [10]:

print("Checking availability of all base geometries...")
missing_bases = []
for base in cfg.bases:
    geojson_path = Path(cfg.geojson_dir) / f"base_{base}.geojson"
    if not geojson_path.exists():
        missing_bases.append(base)
        logging.error(f"Base '{base}' GeoJSON not found at {geojson_path}")
    else:
        print(f"Base '{base}' GeoJSON file exists")
if missing_bases:
    raise FileNotFoundError(f"Missing base geometries: {missing_bases}")
print(f"Feature geometries for all {len(cfg.bases)} features are available.")


Checking availability of all base geometries...
Base 'ft_belvoir' GeoJSON file exists
Base 'ft_custer_training_center' GeoJSON file exists
Base 'ft_hood' GeoJSON file exists
Base 'eglin_afb' GeoJSON file exists
Base 'joint_base_lewis-mcchord' GeoJSON file exists
Base 'vandenberg_afb' GeoJSON file exists
Feature geometries for all 6 features are available.


In [ ]:

print("
" + "=" * 60)
print("Calculating multiresolution hierarchies for all regions...")
print("=" * 60)

resolutions_all = [960, 480, 240, 120, 60, 30]
base_hierarchies = {}

for base in cfg.bases:
    geojson_path = Path(cfg.geojson_dir) / f"base_{base}.geojson"
    gdf_epsg4326 = gpd.read_file(geojson_path).to_crs("EPSG:4326")
    with rasterio.open(cfg.nlcd_img_path) as nlcd:
        gdf_nlcd = gdf_epsg4326.to_crs(nlcd.crs)
        geom_nlcd = gdf_nlcd.geometry.union_all()
        bounds = geom_nlcd.bounds
        plan_levels: list[dict] = []
        coarsest_res = None
        fallback_mask_ratio = 1.0

        for res_m in resolutions_all:
            if res_m < cfg.finest_resolution:
                continue
            ratio = int(round(res_m / 30.0))
            target_size_30m = 128 * ratio
            center_x = (bounds[0] + bounds[2]) / 2
            center_y = (bounds[1] + bounds[3]) / 2
            expanded_bounds = (
                center_x - target_size_30m * nlcd.transform.a / 2,
                center_y - target_size_30m * abs(nlcd.transform.e) / 2,
                center_x + target_size_30m * nlcd.transform.a / 2,
                center_y + target_size_30m * abs(nlcd.transform.e) / 2,
            )
            window = rasterio.windows.from_bounds(*expanded_bounds, transform=nlcd.transform)
            window = window.round_offsets().round_lengths()
            x0 = nlcd.transform.c + window.col_off * nlcd.transform.a
            y0 = nlcd.transform.f + window.row_off * nlcd.transform.e
            mask_30m = rio_rasterize(
                [(geom_nlcd, 1)],
                out_shape=(int(window.height), int(window.width)),
                transform=Affine(nlcd.transform.a, 0, x0, 0, nlcd.transform.e, y0),
                fill=0,
                all_touched=False,
                dtype="uint8",
            ).astype(bool)
            h, w = mask_30m.shape
            if h > target_size_30m:
                start_h = (h - target_size_30m) // 2
                mask_30m = mask_30m[start_h:start_h + target_size_30m, :]
            if w > target_size_30m:
                start_w = (w - target_size_30m) // 2
                mask_30m = mask_30m[:, start_w:start_w + target_size_30m]
            if mask_30m.shape != (target_size_30m, target_size_30m):
                pad_h = target_size_30m - mask_30m.shape[0]
                pad_w = target_size_30m - mask_30m.shape[1]
                mask_30m = np.pad(mask_30m, ((0, pad_h), (0, pad_w)), constant_values=0)
            new_h = mask_30m.shape[0] // ratio
            new_w = mask_30m.shape[1] // ratio
            mask_coarse = mask_30m[:new_h * ratio, :new_w * ratio].reshape(new_h, ratio, new_w, ratio).any(axis=(1, 3))
            mask_ratio = mask_coarse.sum() / mask_coarse.size
            fallback_mask_ratio = mask_ratio
            print(f"{base}: Testing resolution {res_m}m - mask coverage: {mask_ratio:.2%}")
            if mask_ratio < cfg.max_mask_ratio_coarse:
                coarsest_res = res_m
                plan_levels.append({"resolution_m": res_m, "mask_ratio": mask_ratio})
                print(f"{base}: Coarsest resolution {res_m}m (mask: {mask_ratio:.2%})")
                break

        if not plan_levels:
            plan_levels.append({"resolution_m": resolutions_all[0], "mask_ratio": fallback_mask_ratio})
            coarsest_res = resolutions_all[0]
            logging.warning(f"{base}: Using maximum resolution {resolutions_all[0]}m (mask: {fallback_mask_ratio:.2%})")

        finer_candidates = [r for r in resolutions_all if r < coarsest_res and r >= cfg.finest_resolution]
        for res_m in finer_candidates:
            if res_m >= coarsest_res or res_m < cfg.finest_resolution:
                continue
            ratio = int(round(res_m / 30.0))
            target_size_30m = 128 * ratio
            center_x = (bounds[0] + bounds[2]) / 2
            center_y = (bounds[1] + bounds[3]) / 2
            expanded_bounds = (
                center_x - target_size_30m * nlcd.transform.a / 2,
                center_y - target_size_30m * abs(nlcd.transform.e) / 2,
                center_x + target_size_30m * nlcd.transform.a / 2,
                center_y + target_size_30m * abs(nlcd.transform.e) / 2,
            )
            window = rasterio.windows.from_bounds(*expanded_bounds, transform=nlcd.transform)
            window = window.round_offsets().round_lengths()
            x0 = nlcd.transform.c + window.col_off * nlcd.transform.a
            y0 = nlcd.transform.f + window.row_off * nlcd.transform.e
            mask_30m = rio_rasterize(
                [(geom_nlcd, 1)],
                out_shape=(int(window.height), int(window.width)),
                transform=Affine(nlcd.transform.a, 0, x0, 0, nlcd.transform.e, y0),
                fill=0,
                all_touched=False,
                dtype="uint8",
            ).astype(bool)
            h, w = mask_30m.shape
            if h > target_size_30m:
                start_h = (h - target_size_30m) // 2
                mask_30m = mask_30m[start_h:start_h + target_size_30m, :]
            if w > target_size_30m:
                start_w = (w - target_size_30m) // 2
                mask_30m = mask_30m[:, start_w:start_w + target_size_30m]
            if mask_30m.shape != (target_size_30m, target_size_30m):
                pad_h = target_size_30m - mask_30m.shape[0]
                pad_w = target_size_30m - mask_30m.shape[1]
                mask_30m = np.pad(mask_30m, ((0, pad_h), (0, pad_w)), constant_values=0)
            new_h = mask_30m.shape[0] // ratio
            new_w = mask_30m.shape[1] // ratio
            mask_coarse = mask_30m[:new_h * ratio, :new_w * ratio].reshape(new_h, ratio, new_w, ratio).any(axis=(1, 3))
            mask_ratio = mask_coarse.sum() / mask_coarse.size
            if mask_ratio < cfg.max_mask_ratio_fine:
                plan_levels.append({"resolution_m": res_m, "mask_ratio": mask_ratio})
                print(f"{base}: Added finer resolution {res_m}m (mask: {mask_ratio:.2%})")
            else:
                print(f"{base}: Skipping {res_m}m (mask: {mask_ratio:.2%} > {cfg.max_mask_ratio_fine:.0%})")
                break

    base_hierarchies[base] = {"levels": plan_levels, "transforms": {}}

print("Resolution selection complete. Summary:")
for base in cfg.bases:
    level_str = " -> ".join([f"{level['resolution_m']}m ({level['mask_ratio']:.1%})" for level in base_hierarchies[base]['levels']])
    print(f"  {base}: {level_str}")
print("=" * 60 + "
")


In [ ]:

data_npz_path = Path(cfg.data_npz)
npz_data = np.load(data_npz_path)
decode_table = npz_data["decode_table"]
print(f"Loaded decode_table with shape {decode_table.shape} from {data_npz_path}")

D = decode_table.shape[1]
if D != decode_table.shape[2]:
    raise ValueError(f"decode_table must have square patches, got {decode_table.shape[1]}x{decode_table.shape[2]}")
if D != 2:
    raise ValueError(f"Expected tokenizer downsample ratio D=2, found D={D}")

device = torch.device(cfg.device if torch.cuda.is_available() else "cpu")
if cfg.test_mode:
    device = torch.device("cpu")

if cfg.test_mode:
    print("TEST MODE: Using random tokens instead of model inference")
    model = None
else:
    conf = OmegaConf.load(cfg.config_yaml)
    model = instantiate_from_config(conf.ar_model)
    model = model.to(device)
    st_path = Path(cfg.ckpt_dir) / "model.safetensors"
    if st_path.exists():
        from safetensors.torch import load_file
        print(f"Loading model weights from {st_path}")
        state_dict = load_file(str(st_path))
        model.load_state_dict(state_dict, strict=True)
        print("Weights loaded (safetensors)")
    else:
        from accelerate import Accelerator
        mixed_precision = "no" if device.type == "cpu" else "bf16"
        accelerator = Accelerator(mixed_precision=mixed_precision)
        model = accelerator.prepare(model)
        print(f"Loading accelerate state from {cfg.ckpt_dir}")
        accelerator.load_state(cfg.ckpt_dir)
        model = accelerator.unwrap_model(model)
        model = model.to(device)
        print("Weights loaded (accelerate)")
    model.eval()

flat_codes = decode_table.reshape(decode_table.shape[0], -1)
mask_forbidden = np.isin(flat_codes, np.array(cfg.forbidden_nlcd, dtype=flat_codes.dtype)).any(axis=1)
disallowed_tokens = np.where(mask_forbidden)[0].astype(np.int64)
print(f"Identified {len(disallowed_tokens)} tokens that include NLCD classes {cfg.forbidden_nlcd}")

decode_flat = flat_codes
disallowed_tokens_tensor = None if cfg.test_mode else torch.from_numpy(disallowed_tokens).to(device)

majority_values_per_token = []
tokens_by_majority = {}
for token_idx in range(decode_table.shape[0]):
    token_flat = decode_table[token_idx].flatten()
    values, counts = np.unique(token_flat, return_counts=True)
    max_count = counts.max()
    majority = values[counts == max_count]
    majority_set = set(int(v) for v in majority)
    majority_values_per_token.append(majority_set)
    for val in majority_set:
        tokens_by_majority.setdefault(val, set()).add(token_idx)

plot_rows = []


In [ ]:

with rasterio.open(cfg.nlcd_img_path) as nlcd:
    for base in cfg.bases:
        base_samples_dir = inpaint_dir / base
        base_samples_dir.mkdir(parents=True, exist_ok=True)
        base_consensus_dir = consensus_dir / base
        base_consensus_dir.mkdir(parents=True, exist_ok=True)
        suffix = "_test" if cfg.test_mode else ""
        print("
" + "=" * 60)
        print(f"Processing base '{base}'")
        print("=" * 60)

        plan = base_hierarchies[base]
        levels = plan["levels"]
        plan_transforms = plan["transforms"]
        final_res_value = levels[-1]["resolution_m"]

        geojson_path = Path(cfg.geojson_dir) / f"base_{base}.geojson"
        gdf_epsg4326 = gpd.read_file(geojson_path).to_crs("EPSG:4326")
        gdf_nlcd = gdf_epsg4326.to_crs(nlcd.crs)
        geom_nlcd = gdf_nlcd.geometry.union_all()
        bounds = geom_nlcd.bounds
        width_geom = bounds[2] - bounds[0]
        height_geom = bounds[3] - bounds[1]
        center_x_geom = (bounds[0] + bounds[2]) / 2
        center_y_geom = (bounds[1] + bounds[3]) / 2

        samples_list = []
        last_raw = None
        last_mask = None
        last_profile = None

        for sample_idx in range(cfg.samples_per_base):
            torch.manual_seed(cfg.seed + sample_idx)
            print(f"
{base}: Starting sample {sample_idx + 1}/{cfg.samples_per_base}")
            previous_result = None
            previous_res = None
            raw_final = None
            mask_final = None
            profile_final = None

            for level_idx, level in enumerate(levels):
                res_m = level["resolution_m"]
                mask_ratio = level["mask_ratio"]
                print(f"{base} Sample {sample_idx + 1}: Level {level_idx + 1}/{len(levels)} - Resolution {res_m}m (mask: {mask_ratio:.2%})")

                ratio = int(round(res_m / 30.0))
                window_tokens_count = cfg.window_tokens

                expanded_bounds_large = (
                    center_x_geom - width_geom * 5,
                    center_y_geom - height_geom * 5,
                    center_x_geom + width_geom * 5,
                    center_y_geom + height_geom * 5,
                )
                window_expanded = rasterio.windows.from_bounds(*expanded_bounds_large, transform=nlcd.transform)
                window_expanded = window_expanded.round_offsets().round_lengths()
                raw_30m_large = nlcd.read(1, window=window_expanded, boundless=True, fill_value=0)

                width_tokens_needed = int(np.ceil(width_geom / float(res_m)))
                height_tokens_needed = int(np.ceil(height_geom / float(res_m)))
                min_tokens = 128
                buffer_tokens = 8
                target_tokens = max(min_tokens, max(width_tokens_needed, height_tokens_needed) + buffer_tokens)
                target_size_30m = target_tokens * ratio
                print(f"{base}: window footprint {target_tokens} tokens ({target_size_30m} px @30m) for {res_m}m level")

                window_orig = rasterio.windows.from_bounds(*bounds, transform=nlcd.transform)
                rel_col = int(window_orig.col_off - window_expanded.col_off)
                rel_row = int(window_orig.row_off - window_expanded.row_off)
                rel_width = int(window_orig.width)
                rel_height = int(window_orig.height)
                mask_center_x = rel_col + rel_width // 2
                mask_center_y = rel_row + rel_height // 2

                x_start = max(0, mask_center_x - target_size_30m // 2)
                y_start = max(0, mask_center_y - target_size_30m // 2)
                x_end = min(raw_30m_large.shape[1], x_start + target_size_30m)
                y_end = min(raw_30m_large.shape[0], y_start + target_size_30m)
                if x_end - x_start < target_size_30m:
                    x_start = max(0, x_end - target_size_30m)
                if y_end - y_start < target_size_30m:
                    y_start = max(0, y_end - target_size_30m)

                raw_30m = raw_30m_large[y_start:y_end, x_start:x_end]
                if raw_30m.shape != (target_size_30m, target_size_30m):
                    pad_h = target_size_30m - raw_30m.shape[0]
                    pad_w = target_size_30m - raw_30m.shape[1]
                    raw_30m = np.pad(raw_30m, ((0, pad_h), (0, pad_w)), constant_values=0)
                print(f"{base}: extracted raw window shape {raw_30m.shape} at 30m (ratio {ratio})")

                actual_col_off = window_expanded.col_off + x_start
                actual_row_off = window_expanded.row_off + y_start
                x0 = nlcd.transform.c + actual_col_off * nlcd.transform.a
                y0 = nlcd.transform.f + actual_row_off * nlcd.transform.e

                mask_30m = rio_rasterize(
                    [(geom_nlcd, 1)],
                    out_shape=raw_30m.shape,
                    transform=Affine(nlcd.transform.a, 0, x0, 0, nlcd.transform.e, y0),
                    fill=0,
                    all_touched=False,
                    dtype="uint8",
                ).astype(bool)

                trimmed_h = (raw_30m.shape[0] // ratio) * ratio
                trimmed_w = (raw_30m.shape[1] // ratio) * ratio
                raw_30m = raw_30m[:trimmed_h, :trimmed_w]
                mask_30m = mask_30m[:trimmed_h, :trimmed_w]

                new_h = raw_30m.shape[0] // ratio
                new_w = raw_30m.shape[1] // ratio
                reshaped = raw_30m.reshape(new_h, ratio, new_w, ratio).swapaxes(1, 2).reshape(new_h * new_w, ratio * ratio)
                block_modes, _ = sp_mode(reshaped, axis=1, keepdims=False)
                raw_array = block_modes.reshape(new_h, new_w).astype(raw_30m.dtype)

                mask_coarse = mask_30m.reshape(new_h, ratio, new_w, ratio).any(axis=(1, 3))
                coarse_coverage = mask_coarse.sum()
                print(f"{base}: coarse mask coverage {coarse_coverage}/{mask_coarse.size} tokens (ratio={coarse_coverage/mask_coarse.size:.2%})")

                pixel_size = float(res_m)
                coarse_transform = Affine(pixel_size, 0, x0, 0, -pixel_size, y0)
                profile = {
                    "driver": "GTiff",
                    "height": raw_array.shape[0],
                    "width": raw_array.shape[1],
                    "count": 1,
                    "dtype": str(raw_array.dtype),
                    "crs": nlcd.crs,
                    "transform": coarse_transform,
                    "compress": "deflate",
                }

                if res_m not in plan_transforms:
                    plan_transforms[res_m] = coarse_transform

                H_raw, W_raw = raw_array.shape
                Hc = (H_raw // D) * D
                Wc = (W_raw // D) * D
                raw_for_tokens = raw_array[:Hc, :Wc]
                mask_for_tokens = mask_coarse[:Hc, :Wc]

                Ht = Hc // D
                Wt = Wc // D
                patches = raw_for_tokens.reshape(Ht, D, Wt, D).transpose(0, 2, 1, 3).reshape(Ht * Wt, D * D)
                tokens_flat = np.empty(Ht * Wt, dtype=np.int32)
                batch_size = 8192
                for batch_start in range(0, patches.shape[0], batch_size):
                    batch_end = min(batch_start + batch_size, patches.shape[0])
                    p = patches[batch_start:batch_end]
                    eq = (p[:, None, :] == decode_flat[None, :, :]).all(axis=2)
                    has_match = eq.any(axis=1)
                    if np.all(has_match):
                        idx = np.argmax(eq, axis=1)
                        tokens_flat[batch_start:batch_end] = idx.astype(np.int32)
                    else:
                        idx = np.empty(p.shape[0], dtype=np.int32)
                        for i in range(p.shape[0]):
                            if has_match[i]:
                                idx[i] = np.argmax(eq[i])
                            else:
                                diff = (decode_flat != p[i][None, :])
                                dist = diff.sum(axis=1)
                                idx[i] = int(np.argmin(dist))
                        tokens_flat[batch_start:batch_end] = idx
                tokens_grid = tokens_flat.reshape(Ht, Wt)

                mask_tokens = mask_for_tokens.reshape(Ht, D, Wt, D).any(axis=(1, 3))

                masked_pixels = int(mask_tokens.sum())
                print(f"{base}: token grid {Ht}x{Wt}, masked {masked_pixels}/{mask_tokens.size} tokens")

                allowed_tokens_dict = None
                upsampled_prev = None

                if previous_result is not None and level_idx > 0:
                    prev_h, prev_w = previous_result.shape
                    curr_h, curr_w = tokens_grid.shape
                    if curr_h % prev_h == 0 and curr_w % prev_w == 0:
                        scale_y = curr_h // prev_h
                        scale_x = curr_w // prev_w
                        if scale_y != scale_x:
                            logging.warning(f"{base}: Non-uniform scaling between {previous_res}m and {res_m}m; skipping constraints")
                        else:
                            scale = max(scale_y, scale_x)
                            upsampled_prev = previous_result if scale == 1 else np.repeat(np.repeat(previous_result, scale, axis=0), scale, axis=1)
                            masked_coarse_tokens = upsampled_prev[mask_tokens]
                            unique_masked = np.unique(masked_coarse_tokens)
                            print(f"{base}: Computing constraints for {len(unique_masked)} unique masked tokens")
                            if unique_masked.size > 0:
                                vals, counts = np.unique(masked_coarse_tokens, return_counts=True)
                                print(f"{base}: coarse token histogram (top 5 shown) {list(zip(vals.tolist(), counts.tolist()))[:5]}")
                                allowed_tokens_dict = {}
                                for coarse_idx in unique_masked:
                                    coarse_majorities = majority_values_per_token[int(coarse_idx)]
                                    candidate_tokens = set()
                                    for maj_val in coarse_majorities:
                                        candidate_tokens.update(tokens_by_majority.get(maj_val, set()))
                                    allowed_tokens_dict[int(coarse_idx)] = np.array(sorted(candidate_tokens), dtype=np.int64)
                                if allowed_tokens_dict:
                                    sample_allowed = [len(allowed_tokens_dict[k]) for k in allowed_tokens_dict]
                                    print(f"{base}: allowed token counts per coarse id (min/median/max) = {int(np.min(sample_allowed))}/{int(np.median(sample_allowed))}/{int(np.max(sample_allowed))}")
                    else:
                        logging.warning(f"{base}: Token grids not integer-scaled between {previous_res}m and {res_m}m; skipping constraints")

                ys, xs = np.where(mask_tokens)
                if len(ys) == 0:
                    logging.warning(f"{base}: No masked tokens at {res_m}m")
                    previous_result = tokens_grid
                    previous_res = res_m
                    raw_final = raw_array
                    mask_final = mask_coarse
                    profile_final = profile
                    continue

                ymin, ymax = ys.min(), ys.max()
                xmin, xmax = xs.min(), xs.max()
                y_max_start = max(Ht - window_tokens_count, 0)
                x_max_start = max(Wt - window_tokens_count, 0)
                y0 = min(max(0, ymin - window_tokens_count // 2), y_max_start)
                y1 = min(max(0, ymax - window_tokens_count // 2), y_max_start)
                x0 = min(max(0, xmin - window_tokens_count // 2), x_max_start)
                x1 = min(max(0, xmax - window_tokens_count // 2), x_max_start)

                if y0 > y1:
                    fallback_y = min(max(ymin, 0), y_max_start)
                    logging.warning(f"{base}: adjusting y-window start to {fallback_y} (was [{y0},{y1}])")
                    y0 = y1 = fallback_y
                if x0 > x1:
                    fallback_x = min(max(xmin, 0), x_max_start)
                    logging.warning(f"{base}: adjusting x-window start to {fallback_x} (was [{x0},{x1}])")
                    x0 = x1 = fallback_x

                stride = cfg.stride_tokens if level_idx == 0 else max(cfg.stride_tokens // 2, 1)
                print(f"{base}: window search y[{y0},{y1}] x[{x0},{x1}] stride {stride}")

                result_tokens = tokens_grid.copy()
                enforce_min_fill = True if level_idx > 0 else False
                attempts = 2 if level_idx > 0 else 1
                total_windows_seen = 0
                processed_windows_total = 0
                skipped_low_context = 0
                skipped_empty = 0
                windows_processed = 0

                for attempt in range(attempts):
                    windows_processed = 0
                    for y in range(y0, y1 + 1, stride):
                        for x in range(x0, x1 + 1, stride):
                            if y + window_tokens_count > Ht or x + window_tokens_count > Wt:
                                continue
                            total_windows_seen += 1
                            window_tokens_view = result_tokens[y:y + window_tokens_count, x:x + window_tokens_count]
                            window_mask_view = mask_tokens[y:y + window_tokens_count, x:x + window_tokens_count]
                            known_ratio = 1.0 - (window_mask_view.sum() / window_mask_view.size)
                            if enforce_min_fill and level_idx > 0 and known_ratio < cfg.min_filled_ratio:
                                skipped_low_context += 1
                                continue
                            if not np.any(window_mask_view):
                                skipped_empty += 1
                                continue

                            flat_mask = window_mask_view.flatten()
                            known_positions = np.where(~flat_mask)[0].astype(np.int64)
                            unknown_positions = np.where(flat_mask)[0].astype(np.int64)
                            known_tokens_vals = window_tokens_view.flatten()[known_positions].astype(np.int64)

                            coarse_constraint_positions = None
                            coarse_constraint_values = None
                            if allowed_tokens_dict is not None and upsampled_prev is not None:
                                coarse_window = upsampled_prev[y:y + window_tokens_count, x:x + window_tokens_count]
                                coarse_constraint_positions = unknown_positions
                                coarse_constraint_values = coarse_window.flatten()[unknown_positions]

                            if cfg.test_mode:
                                valid_tokens = np.arange(decode_table.shape[0], dtype=np.int64)
                                if disallowed_tokens.size > 0:
                                    valid_tokens = np.setdiff1d(valid_tokens, disallowed_tokens)
                                random_tokens = np.random.choice(valid_tokens, size=len(unknown_positions))
                                if coarse_constraint_positions is not None and coarse_constraint_values is not None and allowed_tokens_dict is not None:
                                    position_lookup = {int(p): idx for idx, p in enumerate(coarse_constraint_positions)}
                                    for idx_pos, pos in enumerate(unknown_positions):
                                        lookup = position_lookup.get(int(pos))
                                        if lookup is None:
                                            continue
                                        coarse_val = int(coarse_constraint_values[lookup])
                                        allowed = allowed_tokens_dict.get(coarse_val)
                                        if allowed is not None and len(allowed) > 0:
                                            random_tokens[idx_pos] = np.random.choice(allowed)
                                full_tokens = np.zeros(window_tokens_count * window_tokens_count, dtype=np.int32)
                                full_tokens[known_positions] = known_tokens_vals
                                full_tokens[unknown_positions] = random_tokens
                                gen_indices = torch.from_numpy(full_tokens).unsqueeze(0)
                            else:
                                cond = torch.tensor([0], dtype=torch.long, device=device)
                                kt = torch.from_numpy(known_tokens_vals).to(device)
                                kp = torch.from_numpy(known_positions).to(device)
                                upos = torch.from_numpy(unknown_positions).to(device)
                                allowed_per_position = None
                                if allowed_tokens_dict is not None and coarse_constraint_values is not None:
                                    allowed_per_position = []
                                    for coarse_val in coarse_constraint_values:
                                        allowed = allowed_tokens_dict.get(int(coarse_val))
                                        if allowed is not None and len(allowed) > 0:
                                            filtered = np.setdiff1d(allowed, disallowed_tokens, assume_unique=False)
                                            if len(filtered) > 0:
                                                allowed_per_position.append(torch.as_tensor(filtered, dtype=torch.long, device=device))
                                                continue
                                        allowed_per_position.append(None)
                                    if allowed_per_position:
                                        counts = [int(t.numel()) if t is not None else 0 for t in allowed_per_position]
                                        constrained_positions = sum(1 for c in counts if c > 0)
                                        if counts:
                                            print(f"{base}: constraints at {res_m}m window ({y},{x}) -> positions {constrained_positions}/{len(counts)} with restrictions, token choices (min/median/max) {np.min(counts)}/{np.median(counts)}/{np.max(counts)}")
                                with torch.no_grad():
                                    gen_indices = generate_inpainting(
                                        model=model,
                                        cond=cond,
                                        known_tokens=kt,
                                        known_positions=kp,
                                        unknown_positions=upos,
                                        cfg_scales=cfg.cfg_scales,
                                        temperature=cfg.temperature,
                                        top_k=cfg.top_k,
                                        top_p=cfg.top_p,
                                        disallowed_classes=disallowed_tokens_tensor,
                                        allowed_token_ids_per_position=allowed_per_position,
                                    )

                            gen_window = gen_indices[0].detach().cpu().numpy().reshape(window_tokens_count, window_tokens_count)

                            edge_mask = np.zeros_like(window_mask_view, dtype=bool)
                            edge_mask[0, :] = True
                            edge_mask[-1, :] = True
                            edge_mask[:, 0] = True
                            edge_mask[:, -1] = True
                            known_edge_mask = edge_mask & (~window_mask_view)
                            if np.any(known_edge_mask):
                                edge_changes = int(np.count_nonzero(gen_window[known_edge_mask] != window_tokens_view[known_edge_mask]))
                                print(f"{base}: edge known-token changes in window ({y},{x}) at {res_m}m -> {edge_changes}")

                            if stride < window_tokens_count:
                                for dy in range(window_tokens_count):
                                    for dx in range(window_tokens_count):
                                        if window_mask_view[dy, dx]:
                                            result_tokens[y + dy, x + dx] = gen_window[dy, dx]
                            else:
                                result_tokens[y:y + window_tokens_count, x:x + window_tokens_count] = gen_window

                            windows_processed += 1
                            processed_windows_total += 1

                    if windows_processed > 0 or level_idx == 0:
                        break
                    if level_idx > 0 and enforce_min_fill:
                        logging.warning(f"{base}: relaxing min_filled_ratio ({cfg.min_filled_ratio}) at {res_m}m to process sparse windows")
                        enforce_min_fill = False

                print(f"{base}: window stats at {res_m}m -> considered {total_windows_seen}, processed {processed_windows_total}, skipped_low_context {skipped_low_context}, skipped_empty {skipped_empty}")
                print(f"{base}: Processed {windows_processed} windows at {res_m}m")
                if windows_processed == 0:
                    logging.warning(f"{base}: no windows processed at {res_m}m")
                if windows_processed == 0 and upsampled_prev is not None:
                    logging.warning(f"{base}: No windows processed at {res_m}m; carrying forward coarse prediction")
                    result_tokens = upsampled_prev.copy()
                    try:
                        coarse_detok = detokenize(result_tokens, decode_table)
                        coarse_values, coarse_counts = np.unique(coarse_detok, return_counts=True)
                        coarse_props = {int(v): float(c) / float(coarse_detok.size) for v, c in zip(coarse_values, coarse_counts)}
                        print(f"{base}: coarse fallback class proportions at {res_m}m -> {coarse_props}")
                    except Exception as exc:
                        logging.error(f"{base}: failed to detokenize coarse fallback at {res_m}m: {exc}")

                if masked_pixels > 0:
                    changed = int(np.count_nonzero(result_tokens[mask_tokens] != tokens_grid[mask_tokens]))
                    print(f"{base}: mask delta {changed}/{masked_pixels} tokens at {res_m}m")
                    if changed == 0:
                        logging.warning(f"{base}: masked region unchanged at {res_m}m")

                previous_result = result_tokens
                previous_res = res_m
                raw_final = raw_array
                mask_final = mask_coarse
                profile_final = profile

            if previous_result is None:
                logging.warning(f"{base}: No tokens generated for sample {sample_idx + 1}")
                continue

            detok = detokenize(previous_result, decode_table).astype(raw_final.dtype)
            detok_h, detok_w = detok.shape
            raw_h, raw_w = raw_final.shape
            if (detok_h, detok_w) != (raw_h, raw_w):
                detok_full = np.zeros_like(raw_final)
                detok_full[:detok_h, :detok_w] = detok
                detok = detok_full

            values, counts = np.unique(detok, return_counts=True)
            proportions = {int(v): float(c) / float(detok.size) for v, c in zip(values, counts)}
            print(f"{base}: sample {sample_idx + 1} class proportions -> {proportions}")

            profile_copy = profile_final.copy()
            transform = profile_copy.get("transform")
            if isinstance(transform, tuple):
                transform = Affine(*transform)
            if transform is None:
                raise ValueError("Source transform is required to write GeoTIFF")
            src_crs = profile_copy.get("crs")
            if src_crs is None:
                raise ValueError("Source CRS is required to write GeoTIFF")
            src_height, src_width = detok.shape
            left, bottom, right, top = array_bounds(src_height, src_width, transform)
            dst_crs = "EPSG:3857"
            dst_transform, dst_width, dst_height = calculate_default_transform(
                src_crs, dst_crs, src_width, src_height, left, bottom, right, top
            )
            destination = np.zeros((dst_height, dst_width), dtype=detok.dtype)
            reproject(
                source=detok,
                destination=destination,
                src_transform=transform,
                src_crs=src_crs,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest,
                num_threads=2,
            )
            profile_copy.update({
                "count": 1,
                "dtype": str(destination.dtype),
                "compress": "deflate",
                "transform": dst_transform,
                "width": dst_width,
                "height": dst_height,
                "crs": dst_crs,
            })
            sample_path = base_samples_dir / f"{base}_inpainted_{final_res_value}m_sample{sample_idx + 1}{suffix}.tif"
            with rasterio.open(sample_path, "w", **profile_copy) as dst:
                dst.write(destination, 1)
            print(f"{base}: wrote inpainted sample {sample_idx + 1} to {sample_path}")

            samples_list.append(detok.copy())
            plot_rows.append((f"{base} (s{sample_idx + 1})", final_res_value, raw_final.copy(), mask_final.copy(), detok.copy()))
            last_raw = raw_final.copy()
            last_mask = mask_final.copy()
            last_profile = profile_final.copy()

        if last_raw is None:
            logging.warning(f"{base}: No samples generated; skipping outputs")
            continue

        sample_data = {
            "base": base,
            "resolution_m": final_res_value,
            "raw": last_raw,
            "mask": last_mask,
            "samples": samples_list,
            "profile": last_profile,
        }
        pickle_path = pickle_dir / f"{base}_samples_{final_res_value}m{suffix}.pkl"
        with open(pickle_path, "wb") as f:
            pickle.dump(sample_data, f)
        print(f"{base}: saved {len(samples_list)} samples to {pickle_path}")

        samples_stack = np.stack(samples_list, axis=0)
        consensus, _ = sp_mode(samples_stack, axis=0, keepdims=False)
        consensus = consensus.astype(last_raw.dtype)

        consensus_transform = plan_transforms.get(final_res_value)
        if consensus_transform is None:
            logging.warning(f"{base}: Missing stored transform for {final_res_value}m; falling back to profile transform")
            consensus_transform = last_profile.get("transform")
        profile_for_consensus = last_profile.copy()
        profile_for_consensus["transform"] = consensus_transform

        profile_consensus_copy = profile_for_consensus.copy()
        transform = profile_consensus_copy.get("transform")
        if isinstance(transform, tuple):
            transform = Affine(*transform)
        if transform is None:
            raise ValueError("Source transform is required to write GeoTIFF")
        src_crs = profile_consensus_copy.get("crs")
        if src_crs is None:
            raise ValueError("Source CRS is required to write GeoTIFF")
        src_height, src_width = consensus.shape
        left, bottom, right, top = array_bounds(src_height, src_width, transform)
        dst_crs = "EPSG:3857"
        dst_transform, dst_width, dst_height = calculate_default_transform(
            src_crs, dst_crs, src_width, src_height, left, bottom, right, top
        )
        destination = np.zeros((dst_height, dst_width), dtype=consensus.dtype)
        reproject(
            source=consensus,
            destination=destination,
            src_transform=transform,
            src_crs=src_crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest,
            num_threads=2,
        )
        profile_consensus_copy.update({
            "count": 1,
            "dtype": str(destination.dtype),
            "compress": "deflate",
            "transform": dst_transform,
            "width": dst_width,
            "height": dst_height,
            "crs": dst_crs,
        })
        consensus_path = base_consensus_dir / f"{base}_consensus_{final_res_value}m{suffix}.tif"
        with rasterio.open(consensus_path, "w", **profile_consensus_copy) as dst:
            dst.write(destination, 1)
        print(f"{base}: wrote consensus raster (mode of {len(samples_list)} samples) to {consensus_path}")

        consensus_values, consensus_counts = np.unique(consensus, return_counts=True)
        consensus_props = {int(v): float(c) / float(consensus.size) for v, c in zip(consensus_values, consensus_counts)}
        print(f"{base}: consensus class proportions: {consensus_props}")

        upsampled_path = base_consensus_dir / f"{base}_consensus_100m{suffix}.tif"
        gdal_cmd = (
            f"gdalwarp -overwrite -s_srs EPSG:3857 -t_srs EPSG:3857 "
            f"-tr 100 100 -r near -co COMPRESS=DEFLATE {consensus_path} {upsampled_path}"
        )
        result = subprocess.run(gdal_cmd, shell=True, capture_output=True, text=True)
        if result.returncode == 0:
            print(f"{base}: upsampled consensus raster from {final_res_value}m to 100m resolution -> {upsampled_path}")
        else:
            logging.error(f"{base}: Failed to upsample raster. Error: {result.stderr}")

        plan["transforms"] = plan_transforms


In [ ]:

if plot_rows:
    grouped = {}
    for label, res_m, raw_i, mask_i, pred_i in plot_rows:
        base = label.split(' (s')[0]
        if base not in grouped:
            grouped[base] = {"res": res_m, "raw": raw_i, "mask": mask_i, "preds": []}
        grouped[base]["preds"].append(pred_i)

    bases_list = list(grouped.keys())
    fig, axes = plt.subplots(len(bases_list), 5, figsize=(15, 3 * len(bases_list)), squeeze=False)
    lut = NLCDTokenizer.lut
    keys = np.array(sorted(lut.keys()), dtype=np.int64)
    rgb_vals = np.array([lut[k] for k in keys], dtype=np.float32)

    for i, base in enumerate(bases_list):
        res_m = grouped[base]["res"]
        raw_i = grouped[base]["raw"]
        mask_i = grouped[base]["mask"]
        preds = grouped[base]["preds"]

        rgb_raw = np.zeros((raw_i.shape[0], raw_i.shape[1], 3), dtype=np.float32)
        for k, rgb in zip(keys, rgb_vals):
            match = raw_i == k
            if match.any():
                rgb_raw[match] = rgb
        axes[i, 0].imshow(rgb_raw, interpolation="nearest")
        axes[i, 0].set_title(f"{base} original ({res_m}m)")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(mask_i.astype(np.uint8), cmap="gray", interpolation="nearest")
        axes[i, 1].set_title("mask")
        axes[i, 1].axis("off")

        for j in range(3):
            img = preds[j] if j < len(preds) else preds[-1]
            rgb_pred = np.zeros((img.shape[0], img.shape[1], 3), dtype=np.float32)
            for k, rgb in zip(keys, rgb_vals):
                match = img == k
                if match.any():
                    rgb_pred[match] = rgb
            axes[i, 2 + j].imshow(rgb_pred, interpolation="nearest")
            axes[i, 2 + j].set_title(f"sample {j + 1}")
            axes[i, 2 + j].axis("off")

    plt.tight_layout()
    suffix = "_test" if cfg.test_mode else ""
    fig_path = figures_dir / f"overview_mixed_res{suffix}.png"
    plt.savefig(fig_path, dpi=200)
    print(f"Saved overview plot to {fig_path}")

suffix = "_test" if cfg.test_mode else ""
archive_base = case_root / run_dir.name
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(run_dir))
print(f"Archived case study outputs to {archive_path}")
